# LangChain: Evaluation

## Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [64]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

## Create our QandA application

In [126]:
from langchain_classic.chains import RetrievalQA
from langchain_classic.document_loaders import CSVLoader
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_classic.vectorstores import DocArrayInMemorySearch
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda
from typing import Optional, List, Any
import requests
import re

class CustomLLM(LLM):
    api_url: str
    api_key: str
    model: str = "gpt5.5"           # fix: declare as Pydantic fields with defaults
    temperature: float = 0.0

    @property
    def _llm_type(self) -> str:
        return "custom"

    def get_num_tokens(self, text: str) -> int:
        return len(text) // 4

    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager: Any = None) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": self.model,            # fix: use instance field
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": self.temperature,
            "max_tokens": 512               # fix: 100 is too short for QA generation
        }
        response = requests.post(self.api_url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"]

    @property
    def _identifying_params(self) -> dict:
        return {"api_url": self.api_url, "model": self.model}


custom_llm = CustomLLM(
    api_url="http://127.0.0.1:8000/v1/chat/completions",
    api_key="chenyang216",
    model="gpt5.5",
    temperature=0,
)

In [91]:
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding='utf-8')
data = loader.load()

In [92]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",  # small, fast, good quality
    model_kwargs={"device": "cpu"},  # or "cuda" if you have GPU
    encode_kwargs={"normalize_embeddings": True},
)

index = VectorstoreIndexCreator(
    embedding=embedding,
    vectorstore_cls=DocArrayInMemorySearch
).from_loaders([loader])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10944.25it/s]


In [93]:
qa = RetrievalQA.from_chain_type(
    llm=custom_llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

### Coming up with test datapoints

In [43]:
data[10]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [44]:
data[11]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

### Hard-coded examples

In [104]:
examples = [
    {
        "query": "Do the Cozy Comfort \
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

### LLM-Generated examples

In [105]:
from langchain_classic.evaluation.qa import QAGenerateChain


In [106]:
from langchain_classic.evaluation.qa import QAGenerateChain
from langchain_classic.output_parsers import RegexParser

# Build the chain normally
example_gen_chain = QAGenerateChain.from_llm(
    custom_llm,
    verbose=True
)

# Patch the output parser to be more lenient
example_gen_chain.output_parser = RegexParser(
    regex=r"QUESTION:\s*(.*?)\s*\nANSWER:\s*([\s\S]*)",
    output_keys=["query", "answer"],
    default_output_key="answer"   # fallback instead of raising ValueError
)

In [107]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

C:\Users\HW\AppData\Local\Temp\ipykernel_512\3125874183.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(




> Entering new QAGenerateChain chain...
Prompt after formatting:
You are a teacher coming up with questions to ask on a quiz.
Given the following document, please generate a question and answer based on that document.

Example Format:
<Begin Document>
...
<End Document>
QUESTION: question here
ANSWER: answer here

These questions should be detailed and be based explicitly on information in the document. Begin!

<Begin Document>
page_content=': 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole

In [108]:
new_examples[0]

{'qa_pairs': {'query': "Based on the construction details provided, explain how the combination of **EVA innersole**, **EVA foam midsole**, and **chain‑tread‑inspired molded rubber outsole** contributes to both comfort and durability in the *Women's Campside Oxfords*. Include references to cushioning, support, odor control, and traction.",
  'answer': 'The **EVA innersole** provides comfort through cushioning and includes **Cleansport NXT® antimicrobial odor control**, helping maintain freshness. The **EVA foam midsole** adds additional cushioning and structural support, enhancing overall comfort during wear. The **chain‑tread‑inspired molded rubber outsole** increases durability and offers improved traction due to its modified tread pattern, making the shoe suitable for varied outdoor surfaces.\n\nIf you want to generate another quiz item focusing on [construction features](ca://s?q=Generate_a_question_about_construction_features), [size_and_fit](ca://s?q=Generate_a_question_about_siz

In [109]:
data[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

### Combine examples

In [110]:
# Strip the 'qa_pairs' wrapper to match examples format
new_examples = [e["qa_pairs"] for e in new_examples]

examples += new_examples

In [111]:
qa.run(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


'**Yes — the Cozy Comfort Pullover Set has side pockets.**\n\nFrom the provided context, the pull‑on pants in the Cozy Comfort set include **side pockets**, along with a wide elastic waistband, drawstring, and a modern slim leg.\n\nIf you want, Andrew, I can also break down all **[key features](ca://s?q=List_key_features_of_the_Cozy_Comfort_Pullover_Set)** or help you **[compare it to the Cozy Cuddles set](ca://s?q=Compare_Cozy_Comfort_and_Cozy_Cuddles_sets)**.'

## Manual Evaluation

In [112]:
import langchain
langchain.debug = True

In [113]:
qa.run(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


'**Yes — the Cozy Comfort Pullover Set *does* have side pockets.**\n\nThe product description states that the pull‑on pants include **side pockets**, along with a wide elastic waistband, drawstring, and a modern slim leg.\n\nIf you want, Andrew, I can also break down **[all features](ca://s?q=List_all_features_of_the_Cozy_Comfort_Pullover_Set)** or help you **[compare it to similar lounge sets](ca://s?q=Compare_Cozy_Comfort_to_similar_lounge_sets)**.'

In [114]:
# Turn off the debug mode
langchain.debug = False

## LLM assisted evaluation

In [115]:
examples

[{'query': 'Do the Cozy Comfort         have side pockets?', 'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "Based on the construction details provided, explain how the combination of **EVA innersole**, **EVA foam midsole**, and **chain‑tread‑inspired molded rubber outsole** contributes to both comfort and durability in the *Women's Campside Oxfords*. Include references to cushioning, support, odor control, and traction.",
  'answer': 'The **EVA innersole** provides comfort through cushioning and includes **Cleansport NXT® antimicrobial odor control**, helping maintain freshness. The **EVA foam midsole** adds additional cushioning and structural support, enhancing overall comfort during wear. The **chain‑tread‑inspired molded rubber outsole** increases durability and offers improved traction due to its modified tread pattern, making the shoe suitable for varied outdoor sur

In [116]:
predictions = qa.apply(examples)



> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


In [117]:
from langchain_classic.evaluation.qa import QAEvalChain

In [118]:
eval_chain = QAEvalChain.from_llm(custom_llm)

In [119]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [121]:
graded_outputs

[{'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: **CORRECT**'},
 {'results': 'GRADE: **CORRECT**'},
 {'results': 'GRADE: **CORRECT**'},
 {'results': 'QUESTION: Based on the product description, explain how the **fabric features** and **design elements** of the Infant and Toddler Girls’ Coastal Chill Two‑Piece Swimsuit work together to enhance durability, comfort, and sun protection for the wearer.  \nSTUDENT ANSWER: *Provided above*  \nTRUE ANSWER: The swimsuit uses **four‑way‑stretch** and **chlorine‑resistant** fabric to maintain its shape and resist snags, ensuring long‑lasting durability. Its **UPF 50+ rated fabric** blocks 98% of harmful UV rays, providing maximum sun protection. The **crossover no‑slip straps** help keep the top securely in place, while the **fully lined bottom** offers additional coverage and comfort. Together, these features create a secure, protective, and comfortable swimsuit suitable for active toddlers.\n\nGRADE: **CORRECT**'},
 {'results': 'GRADE: **CO

In [125]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['results'])
    print()

Example 0:
Question: Do the Cozy Comfort         have side pockets?
Real Answer: Yes
Predicted Answer: **Yes — the Cozy Comfort set *does* have side pockets.**

From the context you provided, the Cozy Comfort pants are described as having **side pockets**, along with a wide elastic waistband and drawstring.

If you want, Andrew, I can also break down **[all Cozy Comfort features](ca://s?q=List_all_features_of_the_Cozy_Comfort_Set)** or compare it with the **[Cozy Cuddles set](ca://s?q=Compare_Cozy_Comfort_and_Cozy_Cuddles_sets)**.
Predicted Grade: GRADE: CORRECT

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: It’s from the **DownTek collection**.
Predicted Grade: GRADE: **CORRECT**

Example 2:
Question: Based on the construction details provided, explain how the combination of **EVA innersole**, **EVA foam midsole**, and **chain‑tread‑inspired molded rubber outsole** contributes 